In [ ]:
"""
Evaluation Script for Day2Night Translation
Calculates FID, LPIPS, and generates qualitative analysis
"""

import torch
import lpips
from pathlib import Path
from PIL import Image
import torchvision.transforms as transforms
import numpy as np
from tqdm import tqdm
import json

class ImageTranslationEvaluator:
    def __init__(self, device='cuda'):
        self.device = device if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {self.device}")

        # Initialize LPIPS
        self.lpips_fn = lpips.LPIPS(net='alex').to(self.device)

        # Image transform
        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def calculate_lpips(self, real_folder, fake_folder, save_per_image=False):
        """
        Calculate LPIPS (Learned Perceptual Image Patch Similarity)
        Lower is better (more similar)
        """
        print("\n📊 Calculating LPIPS...")

        real_images = sorted(Path(real_folder).glob('*.png'))
        fake_images = sorted(Path(fake_folder).glob('*.png'))

        if len(real_images) != len(fake_images):
            print(f"⚠ Warning: Different number of images ({len(real_images)} vs {len(fake_images)})")
            # Match by filename
            real_dict = {img.name: img for img in real_images}
            fake_dict = {img.name: img for img in fake_images}
            common_names = set(real_dict.keys()) & set(fake_dict.keys())
            real_images = [real_dict[name] for name in sorted(common_names)]
            fake_images = [fake_dict[name] for name in sorted(common_names)]

        distances = []
        per_image_results = []

        for real_path, fake_path in tqdm(zip(real_images, fake_images), total=len(real_images)):
            try:
                real_img = self.transform(Image.open(real_path).convert('RGB')).unsqueeze(0).to(self.device)
                fake_img = self.transform(Image.open(fake_path).convert('RGB')).unsqueeze(0).to(self.device)

                with torch.no_grad():
                    d = self.lpips_fn(real_img, fake_img).item()

                distances.append(d)

                if save_per_image:
                    per_image_results.append({
                        'image': real_path.name,
                        'lpips': d
                    })
            except Exception as e:
                print(f"Error processing {real_path.name}: {e}")

        avg_lpips = np.mean(distances)
        std_lpips = np.std(distances)

        print(f"✓ LPIPS Score: {avg_lpips:.4f} ± {std_lpips:.4f}")
        print(f"  (Lower is better - measures perceptual similarity)")

        results = {
            'mean': float(avg_lpips),
            'std': float(std_lpips),
            'min': float(np.min(distances)),
            'max': float(np.max(distances)),
        }

        if save_per_image:
            results['per_image'] = per_image_results

        return results

    def calculate_fid(self, real_folder, fake_folder):
        """
        Calculate FID (Fréchet Inception Distance)
        Lower is better (more similar distributions)
        Requires: pip install pytorch-fid
        """
        print("\n📊 Calculating FID...")

        try:
            from pytorch_fid import fid_score

            paths = [str(real_folder), str(fake_folder)]

            fid_value = fid_score.calculate_fid_given_paths(
                paths,
                batch_size=50,
                device=self.device,
                dims=2048
            )

            print(f"✓ FID Score: {fid_value:.2f}")
            print(f"  (Lower is better - measures distribution similarity)")

            return {'fid': float(fid_value)}

        except ImportError:
            print("⚠ pytorch-fid not installed. Run: pip install pytorch-fid")
            return None
        except Exception as e:
            print(f"✗ Error calculating FID: {e}")
            return None

    def analyze_brightness(self, real_folder, fake_folder):
        """
        Analyze brightness statistics (day vs night)
        Night images should be darker
        """
        print("\n📊 Analyzing brightness...")

        def get_brightness(folder):
            images = list(Path(folder).glob('*.png'))
            brightnesses = []

            for img_path in tqdm(images[:100], desc="Processing"):  # Sample 100 images
                img = Image.open(img_path).convert('L')  # Grayscale
                brightness = np.array(img).mean()
                brightnesses.append(brightness)

            return np.array(brightnesses)

        real_brightness = get_brightness(real_folder)
        fake_brightness = get_brightness(fake_folder)

        print(f"✓ Original (Day) brightness: {real_brightness.mean():.2f} ± {real_brightness.std():.2f}")
        print(f"✓ Generated (Night) brightness: {fake_brightness.mean():.2f} ± {fake_brightness.std():.2f}")
        print(f"✓ Darkness increase: {real_brightness.mean() - fake_brightness.mean():.2f}")

        return {
            'original_mean': float(real_brightness.mean()),
            'original_std': float(real_brightness.std()),
            'generated_mean': float(fake_brightness.mean()),
            'generated_std': float(fake_brightness.std()),
            'darkness_increase': float(real_brightness.mean() - fake_brightness.mean())
        }

    def identify_best_worst(self, per_image_results, top_n=5):
        """
        Identify best and worst translations based on LPIPS
        """
        sorted_results = sorted(per_image_results, key=lambda x: x['lpips'])

        print(f"\n🏆 Best {top_n} translations (lowest LPIPS):")
        for i, result in enumerate(sorted_results[:top_n], 1):
            print(f"  {i}. {result['image']}: {result['lpips']:.4f}")

        print(f"\n⚠ Worst {top_n} translations (highest LPIPS):")
        for i, result in enumerate(sorted_results[-top_n:], 1):
            print(f"  {i}. {result['image']}: {result['lpips']:.4f}")

        return {
            'best': sorted_results[:top_n],
            'worst': sorted_results[-top_n:]
        }

def main():
    import argparse

    parser = argparse.ArgumentParser(description='Evaluate Day2Night Translation')
    parser.add_argument('--real_folder', type=str,
                       default='datasets/cityscapes_test/testA',
                       help='Folder with original day images')
    parser.add_argument('--fake_folder', type=str,
                       default='results/day2night_pretrained/test_latest/images/fake_B',
                       help='Folder with generated night images')
    parser.add_argument('--real_night_folder', type=str, default=None,
                       help='(Optional) Folder with real night images for FID')
    parser.add_argument('--output', type=str, default='evaluation_results.json',
                       help='Output file for results')
    parser.add_argument('--device', type=str, default='cuda',
                       help='Device to use (cuda/cpu)')

    args = parser.parse_args()

    print("=" * 60)
    print("Day2Night Translation Evaluation")
    print("=" * 60)

    # Check folders exist
    if not Path(args.real_folder).exists():
        print(f"✗ Error: Real folder not found: {args.real_folder}")
        return

    if not Path(args.fake_folder).exists():
        print(f"✗ Error: Fake folder not found: {args.fake_folder}")
        return

    # Initialize evaluator
    evaluator = ImageTranslationEvaluator(device=args.device)

    results = {}

    # LPIPS (perceptual similarity between input and output)
    lpips_results = evaluator.calculate_lpips(
        args.real_folder,
        args.fake_folder,
        save_per_image=True
    )
    results['lpips'] = lpips_results

    # Identify best/worst translations
    if 'per_image' in lpips_results:
        best_worst = evaluator.identify_best_worst(lpips_results['per_image'])
        results['best_worst'] = best_worst

    # FID (if real night images provided)
    if args.real_night_folder and Path(args.real_night_folder).exists():
        fid_results = evaluator.calculate_fid(args.real_night_folder, args.fake_folder)
        if fid_results:
            results['fid'] = fid_results
    else:
        print("\n⚠ No real night images provided. Skipping FID calculation.")
        print("  To calculate FID, provide --real_night_folder with real night images")

    # Brightness analysis
    brightness_results = evaluator.analyze_brightness(args.real_folder, args.fake_folder)
    results['brightness'] = brightness_results

    # Save results
    with open(args.output, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"\n💾 Results saved to: {args.output}")

    # Summary
    print("\n" + "=" * 60)
    print("EVALUATION SUMMARY")
    print("=" * 60)
    print(f"LPIPS (perceptual quality): {lpips_results['mean']:.4f} ± {lpips_results['std']:.4f}")
    if 'fid' in results:
        print(f"FID (distribution match): {results['fid']['fid']:.2f}")
    print(f"Brightness reduction: {brightness_results['darkness_increase']:.2f}")
    print("\nInterpretation:")
    print("- LPIPS < 0.3: Good perceptual quality")
    print("- FID < 50: Good distribution match")
    print("- Brightness should decrease significantly for day→night")
    print("=" * 60)

if __name__ == '__main__':
    main()